In [1]:
import json
from pathlib import Path

# Path to your JSON file
json_path = Path("data/thesis_data_1778418853489.json")

# Load the JSON
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("JSON loaded successfully!")
print(type(data))

JSON loaded successfully!
<class 'list'>


In [2]:
import pprint

pp = pprint.PrettyPrinter(depth=3)

pp.pprint(data[0])  # or data if dict

{'fastBuilderMode': 'EXTRASHORT',
 'idAnonymous': 1,
 'minecraftProtocolVersion': 47,
 'stepData': [{'blocksChanged': [],
               'clicks': 1,
               'itemInHand': {...},
               'ping': 168,
               'playerPosition': {...},
               'sneaking': False},
              {'blocksChanged': [...],
               'clicks': 0,
               'itemInHand': {...},
               'ping': 168,
               'playerPosition': {...},
               'sneaking': False},
              {'blocksChanged': [...],
               'clicks': 2,
               'itemInHand': {...},
               'ping': 168,
               'playerPosition': {...},
               'sneaking': False},
              {'blocksChanged': [],
               'clicks': 1,
               'itemInHand': {...},
               'ping': 168,
               'playerPosition': {...},
               'sneaking': False},
              {'blocksChanged': [...],
               'clicks': 0,
               'itemInHand': 

Ok so that worked. The problem now is, as we can see from the pretty print, that each sample has some nested stepData inside. This means we can't just put it in a dataframe and need to process it first.

In [3]:
import pandas as pd

# Create a simplified version of the data
simplified_data = []

for item in data:
    simplified_item = item.copy()

    # Create a new column with the number of steps
    simplified_item["numSteps"] = len(item.get("stepData", []))

    # Remove the original huge stepData field
    del simplified_item["stepData"]

    simplified_data.append(simplified_item)

# Convert to DataFrame
df = pd.DataFrame(simplified_data)

# Preview
print(df.head())

   idAnonymous fastBuilderMode  minecraftProtocolVersion  numSteps
0            1      EXTRASHORT                        47        89
1            2           SHORT                       774       120
2            3      EXTRASHORT                        47       186
3            4      EXTRASHORT                       736       262
4            5           SHORT                        47        81


Ok things are at least graspable now. Ofc this isn't helpful for the project, so lets see what we wanna do with the data.

Next we should crop parts of the steps that have constant size, because neural networks work better with that (or maybe can only work with that even idk)

In [4]:
import random

WINDOW_SIZE = 80

samples = []

for item in data:

    step_data = item.get("stepData", [])

    # Skip sequences that are too short
    if len(step_data) < WINDOW_SIZE:
        continue

    # Pick random starting point
    max_start = len(step_data) - WINDOW_SIZE
    start_index = random.randint(0, max_start)

    # Extract fixed-size window
    window = step_data[start_index:start_index + WINDOW_SIZE]

    samples.append({
        "fastBuilderMode": item["fastBuilderMode"],
        "idAnonymous": item["idAnonymous"],
        "window": window
    })

print("Number of samples:", len(samples))

# TESTING ONLY
df2 = pd.DataFrame(samples)
df2["window"] = df2["window"].apply(len)
print(df2.head())

Number of samples: 9040
  fastBuilderMode  idAnonymous  window
0      EXTRASHORT            1      80
1           SHORT            2      80
2      EXTRASHORT            3      80
3      EXTRASHORT            4      80
4           SHORT            5      80


maybe really for sure what about now?